# SwarmSort Phase 4a: PSO search (Arm C)

Arm C tunes the six training hyperparameters (lr0, lrf, momentum, weight_decay, mosaic, scale) with the
hand-written particle swarm optimizer in `src/pso.py`: **8 particles x 4 rounds = 32 proxy trainings** (seed 42).
The first round is the random initial swarm; every later round moves each particle towards its own best position
and the swarm's best position.

Every proxy training is 12 epochs on the stratified 40% subset of the train split at imgsz 416, and its val mAP@50 is
the fitness. Phase 2 showed that this proxy ranks configurations the same way as 40-epoch training (Spearman
rho = 0.976, n = 8), so the default 12 proxy epochs are used. Random search (Arm B) has the same 32-training budget in `phase4b_random_search.ipynb`; the two notebooks can run at the same
time in separate Kaggle sessions.

**Expected time:** about 2-2.5 hours (one proxy training takes about 4 minutes).

Output: `/kaggle/working/SwarmSort/results/search_pso.json` (every evaluation) and `results/best_pso.json` (the best
configuration, which `src/train_final.py --config` trains fully in Phase 5); proxy run folders go to `results/runs/`.

**Settings:** Accelerator = **GPU T4 x2**, the same as in Phase 2 (keep it for every phase, so all results come from
the same hardware), Internet = on (for the clone and pip), Input = the dataset `viswaprakash1990/garbage-detection`
(*Garbage Detection - 6 Waste Categories*). Run it with **Save Version -> Save & Run All (Commit)** so it keeps
running with the browser closed (up to 12 hours).

### Resuming after a dead session

`search_pso.json` is rewritten after every evaluation. On a restart `search.py` replays the PSO from its seed, reuses every logged
result instead of training again and continues at the first missing evaluation, so a dead session loses at most one
proxy training. To continue: open the notebook, **Add Input -> Notebook Output** and pick the previous (failed or
timed-out) version of this notebook, then run it again. The resume cell below copies that version's
`results/search_pso.json` into place before the search starts. Do not change the search command: a log made with
other settings is refused.

In [ ]:
import os
REPO = "/kaggle/working/SwarmSort"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Shreyansh303/SwarmSort.git {REPO}
%cd {REPO}
# Kaggle's own CUDA build of torch is kept: only Ultralytics is installed, at the version pinned in requirements.txt
!pip install -q ultralytics==8.4.169
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
# Rebuild configs/data.yaml and configs/lists/ with Kaggle paths from the committed configs/split.csv
!python src/verify_dataset.py --root /kaggle/input
assert os.path.exists("configs/data.yaml"), "verify_dataset.py did not write configs/data.yaml"
print(open("configs/data.yaml").read())

In [ ]:
# Resume: reuse the search log of a previous version of this notebook if one is attached as input
import glob, json, shutil, sys
sys.path.insert(0, "src")
from search import make_settings, parse_args
INPUT, LOG = "/kaggle/input", "results/search_pso.json"
expected = make_settings(parse_args(["--method", "pso", "--device", "0"]))  # the settings search.py checks when it resumes
if os.path.exists(LOG):
    print(LOG, "already present: the search continues from it")
else:
    found = []
    for path in glob.glob(f"{INPUT}/**/{LOG}", recursive=True):
        try:
            previous = json.load(open(path))
        except (OSError, ValueError):
            print("Skipping unreadable", path)
            continue
        settings = previous.get("settings", {})
        diff = sorted(k for k in expected.keys() | settings.keys() if settings.get(k) != expected.get(k))
        if diff:  # e.g. a smoke run, another seed or other proxy epochs: search.py would refuse it
            print(f"Skipping {path}: made with other settings ({', '.join(diff)})")
            continue
        found.append((len(previous.get("evaluations", [])), path))
    if found:
        n, path = max(found)  # if several versions are attached, the one that got furthest
        os.makedirs("results", exist_ok=True)
        shutil.copy(path, LOG)
        print(f"Resuming from {path} ({n} evaluations already done)")
    else:
        print("No usable previous log attached: starting a new search")


In [ ]:
# Stratified 40% subset of the train split -> configs/lists/train_proxy.txt and configs/data_proxy.yaml
!python src/training.py --data configs/data.yaml --fraction 0.4

In [ ]:
# One GPU (device 0), as in the Phase 2 proxy runs; default 12 proxy epochs.
# check=True: if search.py stops with an error (e.g. it refuses a log), this cell fails and the version is marked failed
import os, subprocess, sys
subprocess.run([sys.executable, "src/search.py", "--method", "pso", "--device", "0"], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
import json, os
LOG = "results/search_pso.json"
if not os.path.exists(LOG):
    print(f"{LOG} not found: the search wrote no evaluation (see the output of the search cell)")
else:
    log = json.load(open(LOG))
    s, evals = log["settings"], log["evaluations"]
    budget = s["n_particles"] * s["n_iters"]
    print(f"Evaluations done: {len(evals)} of {budget}" + ("" if log["complete"] else "  (not finished: rerun to continue)"))
    failed = sum(1 for e in evals if e.get("failed"))
    if failed:
        print(f"Failed trainings (recorded as mAP@50 0.0): {failed}")
    ok = [e for e in evals if not e.get("failed")]
    if ok:
        best = max(ok, key=lambda e: e["map50"])
        print(f"Best val mAP@50: {best['map50']:.4f} at evaluation {best['index'] + 1} ({best['label']})")
        for name, value in best["params"].items():
            print(f"  {name:<13} {value:g}")
    print("Best so far after each PSO round:")
    best_so_far = 0.0
    for r in range(1, s["n_iters"] + 1):
        scores = [e["map50"] for e in evals if e["round"] == r]
        if not scores:
            break
        best_so_far = max(best_so_far, *scores)
        note = "" if len(scores) == s["n_particles"] else f"  (round incomplete: {len(scores)} of {s['n_particles']})"
        print(f"  round {r} (evaluations up to {(r - 1) * s['n_particles'] + len(scores)}): {best_so_far:.4f}{note}")
    try:  # the same proxy training with the Ultralytics defaults, from the Phase 2 check
        check = json.load(open("results/proxy_check.json"))
        ref = check["runs"][f"proxy_e{s['proxy_epochs']}_s{s['imgsz']}_defaults"]["map50"]
        print(f"Reference: Ultralytics defaults with the same proxy = {ref:.4f}")
    except (OSError, ValueError, KeyError):
        print("Reference: results/proxy_check.json has no matching defaults proxy run")


### Files to download

From the **Output** tab of the finished version, download these two files from `SwarmSort/results/` and put them in
the local project's `results/` folder:

- `results/search_pso.json`: every evaluation (configuration, val mAP@50, run time, GPU)
- `results/best_pso.json`: the best configuration, for Phase 5 (`src/train_final.py --config results/best_pso.json`)

The proxy run folders in `results/runs/` are not needed. Then plot both arms locally with `python src/plot_search.py`.

### If the session died

Check the summary above: if fewer than 32 evaluations are done, use **Add Input -> Notebook Output** to attach this
version, run the notebook again with the same settings (GPU T4 x2, Internet on), and download the files from the
version that finishes.